# Patient-level privacy: how well are individual patients protected?

Tests on the released files of one level, `tier2_merge_drop3_pre_epic`, against an attacker described in `PRIVACY_THREAT_MODEL.md`. All results are counts and shares; no rows leave the SDE. A result is a statement about this attacker, not a guarantee.

The attacker holds the released files and nothing else from the SDE (no keys, no site labels, no other level). They want to find a known person's record, learn something about them, or link their stays. Four scenarios describe what they already know:

| | What the attacker knows | What they try to do | Test (below) |
|---|---|---|---|
| **S1** | the target's demographics: age band, sex, weight band, diagnosis group, outcome | find the target's record among the released stays | row `m = 0` of the S2 table |
| **S2** | S1 plus *m* exact measurements from the stay (concept, hour since admission, value), as a clinician or parent might remember | single out the target's record | known-observation attack |
| **S3** | two released stays | decide whether they belong to the same child | repeat-stay linkage |
| **S4** | demographics only | learn the target's outcome (attribute disclosure) | outcome diversity |

The tests are generous to the attacker on one point (exact hours and exact values are known) and limited on others (observations are chosen at random, there is no outside data, and the release holds no dates). The notebook runs the tests in the order S2 (with S1), S3, S4.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import patient_report as PR
import anonymise as AN

In [ ]:
derived_dir = open_picu_dir / "derived"
harmonised_dir = derived_dir / "harmonised"
out_root = derived_dir / "anonymised"
reports_dir = derived_dir / "reports"
static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
level = "tier2_merge_drop3_pre_epic"

### S1 and S2: known demographics and known observations

For *m* = 0, 1, 3, 5, 10 known exact observations of 500 randomly chosen stays, how many released stays fit. `m = 0` is S1 (demographics only). `unique` is the share of targets with exactly one candidate; `below_k` is the share with fewer than 10. `att5` repeats the attack with a 5% tolerance on the values.

In [ ]:
att = PR.known_observation_attack(out_root, level)
att5 = PR.known_observation_attack(out_root, level, rel_tol=0.05)
att, att5

### S3: linking repeat stays of one child

For stays that have a sibling stay of the same patient, is a sibling among the nearest 1 or 5 stays in the same split group? `chance` is the hit rate for random guessing and `lift` is `hit` divided by `chance`.

In [ ]:
link = PR.repeat_stay_linkage(out_root, level, static)
link

### S4: outcome diversity

For each stay, the outcomes of the stays that share its quasi-identifiers (its cell). A death in a cell where everyone died has its outcome disclosed to an attacker who knows only the demographics.

In [ ]:
div = PR.outcome_diversity(out_root, level)
div

### S2 on the hourly level

Same attack on `tier2_merge_drop3_pre_epic_hourly` (hours rounded, one row per hour). If uniqueness at m = 3 stays near 99%, coarser time alone does not help and the values are doing the identifying.

In [ ]:
att_h = PR.known_observation_attack(out_root, "tier2_merge_drop3_pre_epic_hourly")
att_h

### S4: which cells disclose the outcome, and is it the PIM band?

The cells (combinations of released quasi-identifiers) where at least 90% of the stays died, largest first. Then the same summary with the PIM band left out of the cell, to see how much of the disclosure comes from it. Cell definitions and counts only; check the output rules before anything leaves the SDE.

In [ ]:
cells = PR.outcome_cells(out_root, level)
print(cells.attrs)
display(cells.head(20))
div_nopim = PR.outcome_diversity(out_root, level, exclude=("pim_band",))
div_nopim

### S3: what drives the linking of repeat stays?

The same nearest-stay test, three ways: demographics only, series means only, and without the split group (the attacker searches all stays). Compare the `hit` and `lift` columns with the combined result above.

In [ ]:
link_static = PR.repeat_stay_linkage(out_root, level, static, use="static")
link_series = PR.repeat_stay_linkage(out_root, level, static, use="series")
link_all = PR.repeat_stay_linkage(out_root, level, static, scope="all")
pd.concat([link, link_static, link_series, link_all])

### S4: does the suppressed value reveal the outcome?

For each field with `(suppressed)` values: how many stays carry it, how many of them died, and what share of all deaths carry it. If most stays with a suppressed value are deaths, the marker itself discloses the outcome.

In [ ]:
PR.suppression_vs_outcome(out_root, level)

In [ ]:
w_cur = PR.outcome_diversity_wildcard(out_root, level, static)

## Table 2 and the cost of k

Following Thoral et al. (AmsterdamUMCdb), each level is judged by the final risk for each adversary: P(access) x P(attempt) x P(re-identification), at most 0.05. The friendly researcher and the rogue insurer are scored on the average risk (mean of 1/candidates over all stays), the rogue researcher on the maximum risk (1/smallest group). A suppressed value matches any value. The cells below build the base level again at k = 2, 5 and 10, each with local suppression followed by deletion of the stays still in a cell below k, then compare the cost (stays deleted, fields suppressed) and the risk.

In [ ]:
base = "tier2_merge_drop3_pre_epic"
k_levels = {}
for k, seed in ((2, 20), (5, 21), (10, 22)):
    cfg = AN.config_from_level(out_root, base, name=f"{base}_k{k}", k=k, delete_below_k=True, seed=seed)
    AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
    k_levels[k] = cfg.name
names = [base] + list(k_levels.values())
PR.suppression_cost(out_root, names)

In [ ]:
PR.adversary_risk(out_root, k_levels[2], static)

In [ ]:
PR.risk_by_level(out_root, names, static)[["level", "k", "adversary", "k_min", "k_median", "l_min", "avg_risk", "max_risk", "p_attempt", "final_risk", "ok"]]

## Summary of the k = 2 level

One table of all tests for the k = 2 level, in the form of the final-risk table of Thoral et al. Gate rows (Table 2 adversaries) are scored against a final risk of 0.05. S3 and S4 are reported with their own measures; S2 is informational because an adversary does not hold exact observations of a child. Then the l-diversity breakdown: the shares of stays whose candidates all show one diagnosis group (none suppressed, some suppressed) and whose candidates are all suppressed.

In [ ]:
lv2 = k_levels[2]
summary = PR.privacy_summary(out_root, lv2, static)
summary

In [ ]:
PR.adversary_risk(out_root, lv2, static)[["adversary", "k_min", "l_min", "l_median", "l1_certain", "l1_some_suppressed", "l_unknown"]]

## k = 2 with l-diversity of at least 2

The level is built again at k = 2 with deletion and with `min_l=2`: after local suppression and deletion, the diagnosis group is suppressed wherever the candidates of an adversary would all show the same diagnosis group, so l-diversity is at least 2 as in AmsterdamUMCdb. The summary then shows all tests for this level, and the cost table compares it with the k = 2 level without the l rule.

In [ ]:
base = "tier2_merge_drop3_pre_epic"
k_levels = {2: "tier2_merge_drop3_pre_epic_k2"}
cfg = AN.config_from_level(out_root, base, name=f"{base}_k2_l2", k=2, delete_below_k=True, min_l=2, seed=23)
AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
lv2l = cfg.name
PR.suppression_cost(out_root, [k_levels[2], lv2l])

In [ ]:
PR.privacy_summary(out_root, lv2l, static)

In [ ]:
PR.adversary_risk(out_root, lv2l, static)[["adversary", "k_min", "l_min", "l_median", "l1_certain", "l1_some_suppressed", "l_unknown"]]

## k counted in patients

The same k = 2 level with l >= 2, built so that k counts different patients (two stays of one child do not make a cell of two). The first table is the existing level seen by patients (what we had, re-scored); the second is the rebuilt level. `repeat_share` is the share of targets whose candidate stays include two or more stays of the same child.

In [ ]:
base = "tier2_merge_drop3_pre_epic"
lv2l = f"{base}_k2_l2"
PR.adversary_risk(out_root, lv2l, static)[["adversary", "k_min", "k_min_stays", "repeat_share", "avg_risk", "max_risk", "final_risk", "ok"]]

In [ ]:
cfg = AN.config_from_level(out_root, base, name=f"{base}_k2_l2_pat", k=2, k_unit="patients", delete_below_k=True, min_l=2, seed=24)
AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
lv2p = cfg.name
PR.suppression_cost(out_root, [lv2l, lv2p])

In [ ]:
PR.privacy_summary(out_root, lv2p, static)

## Table 2 for the release level

The risk table of the release level in the layout of Table 2 of Thoral et al. (AmsterdamUMCdb). k and l in the average-risk block are medians over the stays; in the maximum-risk block they are the smallest. The friendly researcher's P(intention) depends on the population figure given below.

In [ ]:
lv2p = "tier2_merge_drop3_pre_epic_k2_l2_pat"
table2 = PR.risk_table(out_root, lv2p, static)
table2